# FINAL NUMBERS — one run, every candidate

Produces every legitimate "final Table 2" candidate in a single session:

| stage | experiment | what it is | cost |
|---|---|---|---|
| A | `v8_coslr` | full (3,4,7)×5 sweep, baseline recipe + **cosine LR** + best-val checkpoints | 15 training runs (the big part) |
| B | `v8_coslr_best` | the same runs evaluated at their **best-val** epoch | eval-only |
| C1 | `v8_coslr_ens` | mean-softmax **ensemble** of the 5 cosine runs per variant | eval-only |
| C2 | `v2_baseline_ens` | ensemble of the 5 **frozen-baseline** runs per variant (fallback if cosine disappoints) | eval-only |
| C3 | `v8_coslr_bestens` | ensemble of the best-val checkpoints (optional) | eval-only |
| D | — | decision table: all candidates side by side vs the frozen baseline | — |

Fairness is by construction: every candidate applies the identical recipe to all
three variants. Everything is resume-safe (skip-if-exists); if interrupted, Run
All again. Run order matters only in that A must finish before B/C1/C3.

## 0. Imports

In [ ]:
import os, sys, dataclasses
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import amg_pipeline as amg
from amg_pipeline.config import RunConfig, make_run_id
from amg_pipeline import paths

print("amg_pipeline loaded from:", os.path.dirname(amg.__file__))

## 1. CONFIG

In [ ]:
# === EXPERIMENT IDENTITY ===
FINAL_EXPERIMENT    = "v8_coslr"                  # stage A sweep
BASELINE_EXPERIMENT = "v2_yaw_correction-epsV1"   # frozen reference + ensemble fallback source
EXPERIMENTS_ROOT = os.path.join(REPO_ROOT, "experiments")

# === SWEEP SHAPE (full Table-2 fairness: all three variants) ===
CHANNEL_VARIANTS = (3, 4, 7)
WIDTH  = "base"
N_RUNS = 5
SEED   = 42

# === RECIPE: baseline + Candidate 3 only (norm + photo_aug stay OFF) ===
NORM        = "none"
PHOTO_AUG   = "none"
LR_SCHEDULE = "cosine"     # C3: cosine decay; best-val checkpoints are always saved now

# === TRAINING DATA ===
ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_orthomosaics"
NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_normalmaps"
MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\snippets_masks"

# === TEST WALLS ===
TEST_ORTHO_DIR     = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\01_test-images"
TEST_NORMALMAP_DIR = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\02_test-normals"
TEST_MASK_DIR      = r"C:\Users\admin\Desktop\AmG-Artikel2025\PaperVersion2\V2-3_NewNormalmaps_tiltON_epsV1\2025_AmG_TrainingTestingDataCompress\Testing\03_test-masks"
WALLS = ["wall1", "wall2", "wall3", "wall4"]
ORTHO_PATTERN     = "{wall}_png-ortho.png"
NORMALMAP_PATTERN = "{wall}_DEM_normalmap.png"
MASK_PATTERN      = "{wall}_png-ortho.png"

# === HYPER-PARAMS / ROI (unchanged) ===
N_EPOCHS, BATCH_SIZE, LR = 300, 16, 1e-4
ROI_OPERATION, KERNEL_RADIUS = "closing", 45

BASE_CONFIG = RunConfig(
    channels=7, run_number=1,
    experiment_name=FINAL_EXPERIMENT, experiments_root=EXPERIMENTS_ROOT,
    ortho_dir=ORTHO_DIR, normalmap_dir=NORMALMAP_DIR, mask_dir=MASK_DIR,
    test_ortho_dir=TEST_ORTHO_DIR, test_normalmap_dir=TEST_NORMALMAP_DIR,
    test_mask_dir=TEST_MASK_DIR, walls=WALLS,
    ortho_pattern=ORTHO_PATTERN, normalmap_pattern=NORMALMAP_PATTERN, mask_pattern=MASK_PATTERN,
    seed=SEED, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=LR,
    roi_operation=ROI_OPERATION, kernel_radius=KERNEL_RADIUS,
    width_mult=WIDTH, norm=NORM, photo_aug=PHOTO_AUG, lr_schedule=LR_SCHEDULE,
)
print("Config OK:", FINAL_EXPERIMENT, "| lr_schedule:", LR_SCHEDULE,
      "| variants:", CHANNEL_VARIANTS, "| runs:", N_RUNS,
      "=", len(CHANNEL_VARIANTS) * N_RUNS, "training runs")

## 2. Verify architecture & preview — NO training

In [ ]:
amg.verify_architecture()

import glob
print("\n=== TRAINING DIRECTORIES ===")
for label, d in [("ortho", ORTHO_DIR), ("normalmap", NORMALMAP_DIR), ("mask", MASK_DIR)]:
    n = len(glob.glob(os.path.join(d, "*.png"))) if os.path.isdir(d) else None
    print(f"  {'OK ' if n else 'X  '} {label:10s} -> {n if n else 'MISSING'} png")
print("\n=== TEST WALL INPUTS ===")
all_ok = True
for wall in WALLS:
    for label, fn in [("ortho", paths.test_ortho_path(BASE_CONFIG, wall)),
                      ("normal", paths.test_normalmap_path(BASE_CONFIG, wall)),
                      ("mask", paths.test_mask_path(BASE_CONFIG, wall))]:
        ok = os.path.exists(fn); all_ok &= ok
        if not ok: print(f"  X  {wall} {label}: {fn}")
print("  all present" if all_ok else "  *** fix missing inputs before running ***")
print("\n=== BASELINE CHECKPOINTS (for the fallback ensemble) ===")
for ch in CHANNEL_VARIANTS:
    for run_n in range(1, N_RUNS + 1):
        c = dataclasses.replace(BASE_CONFIG, channels=ch, run_number=run_n,
                                experiment_name=BASELINE_EXPERIMENT)
        if not os.path.exists(paths.checkpoint_path(c)):
            print("  X  missing:", paths.checkpoint_path(c)); all_ok = False
print("  all present" if all_ok else "  *** some baseline checkpoints missing ***")

## 3. Stage A — the sweep (15 training runs, overnight)

In [ ]:
DO_RUN_SWEEP = True    # <-- armed
FORCE = False

if DO_RUN_SWEEP:
    amg.run_sweep(BASE_CONFIG, channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                  force=FORCE)
    print("=== STAGE A COMPLETE ===")
else:
    print("DO_RUN_SWEEP is False")

## 4. Stage B — best-val checkpoints, evaluated (eval-only)

Same runs, loaded at their best-validation epoch via `checkpoint_experiment`
(+`checkpoint_filename`); outputs isolated under `v8_coslr_best`.

In [ ]:
DO_RUN_BEST = True     # <-- armed

if DO_RUN_BEST:
    best_base = dataclasses.replace(BASE_CONFIG,
                                    experiment_name=f"{FINAL_EXPERIMENT}_best",
                                    checkpoint_experiment=FINAL_EXPERIMENT,
                                    checkpoint_filename="model_best.pth")
    amg.run_sweep(best_base, channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                  do_train=False, force=FORCE)
    print("=== STAGE B COMPLETE ===")
else:
    print("DO_RUN_BEST is False")

## 5. Stage C — ensembles (eval-only)

Mean-softmax over the 5 runs per variant: the new cosine runs, and the frozen
baseline runs as the fallback candidate. Optional: ensemble of best-val
checkpoints.

In [ ]:
DO_RUN_ENSEMBLES = True    # <-- armed
DO_BEST_ENSEMBLE = False    # optional extra candidate

if DO_RUN_ENSEMBLES:
    amg.run_ensemble(BASE_CONFIG, FINAL_EXPERIMENT, f"{FINAL_EXPERIMENT}_ens",
                     channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS, force=FORCE)
    amg.run_ensemble(BASE_CONFIG, BASELINE_EXPERIMENT, "v2_baseline_ens",
                     channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS, force=FORCE)
    if DO_BEST_ENSEMBLE:
        amg.run_ensemble(BASE_CONFIG, FINAL_EXPERIMENT, f"{FINAL_EXPERIMENT}_bestens",
                         channel_variants=CHANNEL_VARIANTS, n_runs=N_RUNS,
                         checkpoint_filename="model_best.pth", force=FORCE)
    print("=== STAGE C COMPLETE ===")
else:
    print("DO_RUN_ENSEMBLES is False")

## 6. Stage D — decision table

Every candidate side by side: AllWalls mean-stone IoU (mean ± std for 5-run
candidates, single value for ensembles), per-class 7ch, and the 7ch−4ch gap.

In [ ]:
CANDIDATES = [
    ("baseline (frozen, per-run)", BASELINE_EXPERIMENT, "runs"),
    ("cosine (per-run)",           FINAL_EXPERIMENT, "runs"),
    ("cosine best-val (per-run)",  f"{FINAL_EXPERIMENT}_best", "runs"),
    ("baseline ENSEMBLE",          "v2_baseline_ens", "ens"),
    ("cosine ENSEMBLE",            f"{FINAL_EXPERIMENT}_ens", "ens"),
    ("cosine best-val ENSEMBLE",   f"{FINAL_EXPERIMENT}_bestens", "ens"),
]
rows = []
for label, exp, kind in CANDIDATES:
    mp = paths.manifest_path(dataclasses.replace(BASE_CONFIG, experiment_name=exp))
    if not os.path.exists(mp):
        print(f"(missing) {label}: {mp}")
        continue
    mf = pd.read_csv(mp)
    aw = mf[mf.wall == "AllWalls"]
    row = {"candidate": label}
    for ch in (3, 4, 7):
        s = aw[aw.channels == ch]["IoU_mean_stones"]
        if not len(s):
            row[f"{ch}ch"] = "—"
        elif kind == "runs":
            row[f"{ch}ch"] = f"{s.mean():.4f} ± {s.std():.4f}"
        else:
            row[f"{ch}ch"] = f"{s.iloc[0]:.4f}"
    g7 = aw[aw.channels == 7]["IoU_mean_stones"].mean()
    g4 = aw[aw.channels == 4]["IoU_mean_stones"].mean()
    q7 = aw[aw.channels == 7]["IoU_Quarry"].mean()
    row["gap_7-4"] = f"{g7-g4:+.4f}" if not (np.isnan(g7) or np.isnan(g4)) else "—"
    row["7ch_Quarry"] = f"{q7:.4f}" if not np.isnan(q7) else "—"
    row["_sort"] = g7
    rows.append(row)

tbl = pd.DataFrame(rows).sort_values("_sort", ascending=False).drop(columns="_sort")
pd.set_option("display.width", 200)
display(tbl.reset_index(drop=True))
print("\nGuidance: the paper's Table 2 comes from ONE candidate applied to all three")
print("variants; report it transparently (e.g. 'ensemble of 5 runs, cosine schedule').")
print("Prefer the highest 7ch that does not regress 7ch Quarry vs the frozen baseline.")